In [12]:
import gdown
import pandas as pd

file_id = '124UtopVQ_xJLlRoF9_-jn6j04UG-Q3rJ'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'emotion_dataset.csv'
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
display(df.head())

Downloading...
From: https://drive.google.com/uc?id=124UtopVQ_xJLlRoF9_-jn6j04UG-Q3rJ
To: /content/emotion_dataset.csv
100%|██████████| 620k/620k [00:00<00:00, 127MB/s]


,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [13]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    cleaned = [w for w in words if w not in stop_words]
    return " ".join(cleaned)

text_col = 'Text' if 'Text' in df.columns else df.columns[0]
label_col = 'Emotion' if 'Emotion' in df.columns else df.columns[1]

df['cleaned_text'] = df[text_col].apply(clean_text)
display(df[['cleaned_text', label_col]].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,cleaned_text,Emotion
0,seriously hate one subject death feel reluctan...,fear
1,im full life feel appalled,anger
2,sit write start dig feelings think afraid acce...,fear
3,ive really angry r feel like idiot trusting fi...,joy
4,feel suspicious one outside like rapture happe...,fear


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_text'])
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (5937, 5000)


In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_preds = nb.predict(X_test)

svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
svm_preds = svm.predict(X_test)

print("--- Naive Bayes Performance ---")
print(f"Accuracy: {accuracy_score(y_test, nb_preds):.4f}")
print(classification_report(y_test, nb_preds))

print("\n--- SVM Performance ---")
print(f"Accuracy: {accuracy_score(y_test, svm_preds):.4f}")
print(classification_report(y_test, svm_preds))

--- Naive Bayes Performance ---
Accuracy: 0.9099
              precision    recall  f1-score   support

       anger       0.88      0.95      0.91       392
        fear       0.92      0.92      0.92       416
         joy       0.95      0.86      0.90       380

    accuracy                           0.91      1188
   macro avg       0.91      0.91      0.91      1188
weighted avg       0.91      0.91      0.91      1188


--- SVM Performance ---
Accuracy: 0.9478
              precision    recall  f1-score   support

       anger       0.93      0.96      0.94       392
        fear       0.97      0.92      0.94       416
         joy       0.95      0.96      0.96       380

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188

